# CC3074 – Modelización y Simulación
## Métodos numéricos aplicados a simulación: Euler y Runge-Kutta (RK4)

**Universidad del Valle de Guatemala**

Este notebook estudia cómo utilizar métodos numéricos para aproximar la evolución de sistemas continuos descritos mediante ecuaciones diferenciales.

### Objetivos
- Comprender para qué sirven los métodos numéricos en simulación.
- Implementar el método de Euler en Python.
- Implementar Runge-Kutta de cuarto orden (RK4) en Python.
- Comparar Euler, RK4 y una solución exacta.
- Analizar el efecto del tamaño de paso \(h\).
- Interpretar los resultados desde el punto de vista de modelización y simulación.

### Caso de estudio
Utilizaremos la **Ley de Enfriamiento de Newton** para simular la temperatura de una taza de café.


## 1. Del sistema real a la simulación

En un sistema continuo, una variable puede cambiar constantemente con respecto al tiempo.

El flujo que seguiremos es:

**Sistema real → Modelo matemático → Ecuación diferencial → Método numérico → Python → Simulación → Análisis**

Una ecuación diferencial puede escribirse de forma general como:

\[
\frac{dy}{dt}=f(t,y)
\]

La función \(f(t,y)\) indica la tasa de cambio del estado del sistema.


## 2. Modelo: enfriamiento de una taza de café

Supongamos:

- Temperatura inicial: \(T_0=90^\circ C\)
- Temperatura ambiente: \(T_a=20^\circ C\)
- Constante de enfriamiento: \(k=0.1\)

La Ley de Enfriamiento de Newton es:

\[
\frac{dT}{dt}=-k(T-T_a)
\]

Por lo tanto:

\[
\frac{dT}{dt}=-0.1(T-20)
\]

Simularemos los primeros **10 minutos**.


In [ ]:
# Parámetros del modelo
T0 = 90.0
Ta = 20.0
k = 0.1

# Parámetros de simulación
h = 1.0
tiempo_total = 10.0

print(f"Temperatura inicial: {T0} °C")
print(f"Temperatura ambiente: {Ta} °C")
print(f"k: {k}")
print(f"Tamaño de paso h: {h} min")


## 3. Función del modelo

Es conveniente separar el **modelo del sistema** del **método numérico**.

La siguiente función representa:

\[
\frac{dT}{dt}=-k(T-T_a)
\]

Esto permitirá utilizar el mismo modelo tanto con Euler como con RK4.


In [ ]:
def modelo(t, T, k=0.1, Ta=20.0):
    return -k * (T - Ta)

print("Pendiente inicial:", modelo(0, T0, k, Ta), "°C/min")


## 4. Método de Euler

Euler utiliza la pendiente actual para estimar el siguiente estado:

\[
y_{n+1}=y_n+h\,f(t_n,y_n)
\]

Interpretación:

\[
\text{Nuevo estado}=
\text{Estado actual}+
\text{Cambio estimado}
\]

Para el café, en \(t=0\):

\[
f(0,90)=-0.1(90-20)=-7
\]

Con \(h=1\):

\[
T_1=90+(1)(-7)=83^\circ C
\]


In [ ]:
# Primera iteración manual de Euler
t = 0.0
T = T0

pendiente = modelo(t, T, k, Ta)
T_siguiente = T + h * pendiente

print(f"T actual: {T:.2f} °C")
print(f"Pendiente: {pendiente:.2f} °C/min")
print(f"T siguiente: {T_siguiente:.2f} °C")


## 5. Función reutilizable de Euler

Ahora convertimos la fórmula en una función general.


In [ ]:
def euler(f, t, y, h):
    return y + h * f(t, y)


## 6. Simulación completa con Euler


In [ ]:
import numpy as np
import pandas as pd

def simular_euler(T0, Ta, k, h, tiempo_total):
    tiempos = np.arange(0, tiempo_total + h/2, h)
    temperaturas = [T0]
    T = T0

    f = lambda t, T: modelo(t, T, k, Ta)

    for t in tiempos[:-1]:
        T = euler(f, t, T, h)
        temperaturas.append(T)

    return tiempos, np.array(temperaturas)

tiempos_euler, temperaturas_euler = simular_euler(T0, Ta, k, h, tiempo_total)

tabla_euler = pd.DataFrame({
    "Tiempo (min)": tiempos_euler,
    "Euler (°C)": temperaturas_euler
})

tabla_euler


## 7. Gráfica de Euler

La gráfica permite observar la evolución temporal del sistema y verificar si el comportamiento obtenido tiene sentido.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(tiempos_euler, temperaturas_euler, marker="o", label="Euler")
plt.axhline(Ta, linestyle="--", label="Temperatura ambiente")
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Enfriamiento del café - Método de Euler")
plt.grid(True)
plt.legend()
plt.show()


## 8. Runge-Kutta de cuarto orden (RK4)

Euler utiliza una pendiente por paso. RK4 evalúa cuatro pendientes:

\[
k_1=f(t_n,y_n)
\]

\[
k_2=f\left(t_n+\frac{h}{2},y_n+\frac{h}{2}k_1\right)
\]

\[
k_3=f\left(t_n+\frac{h}{2},y_n+\frac{h}{2}k_2\right)
\]

\[
k_4=f(t_n+h,y_n+hk_3)
\]

Luego:

\[
y_{n+1}=y_n+\frac{h}{6}(k_1+2k_2+2k_3+k_4)
\]

La idea no es actualizar definitivamente el sistema cuatro veces. Las cuatro pendientes son evaluaciones utilizadas para obtener una mejor estimación del siguiente estado.


## 9. Primera iteración de RK4 paso a paso


In [ ]:
t = 0.0
T = T0

f = lambda t, T: modelo(t, T, k, Ta)

k1 = f(t, T)
k2 = f(t + h/2, T + h*k1/2)
k3 = f(t + h/2, T + h*k2/2)
k4 = f(t + h, T + h*k3)

T_rk_1 = T + (h/6) * (k1 + 2*k2 + 2*k3 + k4)

print(f"k1 = {k1:.6f}")
print(f"k2 = {k2:.6f}")
print(f"k3 = {k3:.6f}")
print(f"k4 = {k4:.6f}")
print(f"T(1) con RK4 = {T_rk_1:.6f} °C")


## 10. Función reutilizable de RK4


In [ ]:
def rk4(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h/2, y + h*k1/2)
    k3 = f(t + h/2, y + h*k2/2)
    k4 = f(t + h, y + h*k3)

    return y + (h/6) * (k1 + 2*k2 + 2*k3 + k4)


## 11. Simulación completa con RK4


In [ ]:
def simular_rk4(T0, Ta, k, h, tiempo_total):
    tiempos = np.arange(0, tiempo_total + h/2, h)
    temperaturas = [T0]
    T = T0

    f = lambda t, T: modelo(t, T, k, Ta)

    for t in tiempos[:-1]:
        T = rk4(f, t, T, h)
        temperaturas.append(T)

    return tiempos, np.array(temperaturas)

tiempos_rk4, temperaturas_rk4 = simular_rk4(T0, Ta, k, h, tiempo_total)

tabla_rk4 = pd.DataFrame({
    "Tiempo (min)": tiempos_rk4,
    "RK4 (°C)": temperaturas_rk4
})

tabla_rk4


## 12. Solución exacta

Para este caso conocemos la solución analítica:

\[
T(t)=T_a+(T_0-T_a)e^{-kt}
\]

Esto nos permite medir el error de los métodos numéricos.

> En muchos modelos reales no tendremos una solución exacta disponible. Aquí se utiliza con fines didácticos y de validación.


In [ ]:
def solucion_exacta(t, T0, Ta, k):
    return Ta + (T0 - Ta) * np.exp(-k * t)

temperaturas_exactas = solucion_exacta(tiempos_euler, T0, Ta, k)

print(f"T exacta en t=1: {solucion_exacta(1, T0, Ta, k):.6f} °C")


## 13. Euler vs. RK4 vs. solución exacta

Ahora comparamos los tres resultados utilizando el mismo tamaño de paso:

\[
h=1
\]


In [ ]:
comparacion = pd.DataFrame({
    "Tiempo (min)": tiempos_euler,
    "Euler (°C)": temperaturas_euler,
    "RK4 (°C)": temperaturas_rk4,
    "Exacta (°C)": temperaturas_exactas
})

comparacion


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(tiempos_euler, temperaturas_euler, marker="o", label="Euler")
plt.plot(tiempos_rk4, temperaturas_rk4, marker="s", label="RK4")
plt.plot(tiempos_euler, temperaturas_exactas, label="Solución exacta")

plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Euler vs. RK4 vs. solución exacta")
plt.grid(True)
plt.legend()
plt.show()


### Interpretación

Las tres curvas comienzan en \(90^\circ C\) y descienden hacia la temperatura ambiente.

Con \(h=1\):

- Euler presenta una desviación visible.
- RK4 queda prácticamente superpuesto con la solución exacta.
- Que RK4 esté muy cerca de la solución exacta en este ejemplo **no significa que siempre sea exacto**.


## 14. Cálculo del error

Utilizaremos error absoluto:

\[
Error=|T_{exacta}-T_{aproximada}|
\]


In [ ]:
error_euler = np.abs(temperaturas_exactas - temperaturas_euler)
error_rk4 = np.abs(temperaturas_exactas - temperaturas_rk4)

tabla_error = pd.DataFrame({
    "Tiempo (min)": tiempos_euler,
    "Error Euler": error_euler,
    "Error RK4": error_rk4
})

tabla_error


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(tiempos_euler, error_euler, marker="o", label="Error Euler")
plt.plot(tiempos_euler, error_rk4, marker="s", label="Error RK4")
plt.xlabel("Tiempo (min)")
plt.ylabel("Error absoluto (°C)")
plt.title("Error numérico: Euler vs. RK4")
plt.grid(True)
plt.legend()
plt.show()


## 15. ¿Qué ocurre cuando modificamos \(h\)?

El tamaño de paso determina cada cuánto actualizamos el estado de la simulación.

Experimentaremos con:

\[
h=2,\quad 1,\quad 0.5,\quad 0.1
\]

Un paso menor normalmente mejora la aproximación, pero aumenta el número de cálculos.


In [ ]:
valores_h = [2.0, 1.0, 0.5, 0.1]

plt.figure(figsize=(10, 6))

for h_prueba in valores_h:
    tiempos, temperaturas = simular_euler(T0, Ta, k, h_prueba, tiempo_total)
    plt.plot(tiempos, temperaturas, marker="o", label=f"Euler h={h_prueba}")

t_ref = np.linspace(0, tiempo_total, 300)
plt.plot(t_ref, solucion_exacta(t_ref, T0, Ta, k), label="Solución exacta")

plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Efecto del tamaño de paso h en Euler")
plt.grid(True)
plt.legend()
plt.show()


## 16. Error final para diferentes valores de \(h\)

Comparemos el error en \(t=10\).


In [ ]:
resultados_h = []

T_exacta_final = float(solucion_exacta(tiempo_total, T0, Ta, k))

for h_prueba in valores_h:
    _, temp_e = simular_euler(T0, Ta, k, h_prueba, tiempo_total)
    _, temp_r = simular_rk4(T0, Ta, k, h_prueba, tiempo_total)

    resultados_h.append({
        "h": h_prueba,
        "Euler final (°C)": temp_e[-1],
        "Error Euler": abs(T_exacta_final - temp_e[-1]),
        "RK4 final (°C)": temp_r[-1],
        "Error RK4": abs(T_exacta_final - temp_r[-1])
    })

pd.DataFrame(resultados_h)


## 17. ¿Qué aprendemos de la comparación?

### Euler
- Una evaluación de la pendiente por paso.
- Fácil de comprender e implementar.
- Bajo costo por paso.
- Puede necesitar un \(h\) pequeño para lograr buena precisión.

### RK4
- Cuatro evaluaciones de la pendiente por paso.
- Mayor trabajo computacional por paso.
- Generalmente ofrece una aproximación mucho mejor con el mismo \(h\).

La selección del método depende de:

- precisión requerida;
- tamaño de paso;
- estabilidad;
- complejidad del modelo;
- costo computacional;
- objetivo de la simulación.


# 18. Ejercicio integrador

Una bebida comienza a:

\[
T_0=95^\circ C
\]

La temperatura ambiente es:

\[
T_a=22^\circ C
\]

y:

\[
k=0.08
\]

El modelo es:

\[
\frac{dT}{dt}=-0.08(T-22)
\]

Simular los primeros **20 minutos**.

### Parte A
Implementar Euler con \(h=1\).

### Parte B
Implementar RK4 con \(h=1\).

### Parte C
Graficar Euler y RK4.

### Parte D
Repetir utilizando:

\[
h=2,\quad h=0.5
\]

### Parte E
La solución exacta es:

\[
T(t)=22+73e^{-0.08t}
\]

Calcular el error de Euler y RK4 en \(t=20\).

### Preguntas
1. ¿Cómo afecta \(h\) a los resultados?
2. ¿Qué diferencias existen entre Euler y RK4?
3. ¿Cuál requiere más cálculos por paso?
4. ¿Qué sucede con la temperatura conforme aumenta el tiempo?
5. ¿Cuál método queda más cerca de la solución exacta para el mismo \(h\)?


In [ ]:
# ============================================================
# ESPACIO DE TRABAJO - EJERCICIO INTEGRADOR
# ============================================================

T0_ejercicio = 95.0
Ta_ejercicio = 22.0
k_ejercicio = 0.08
tiempo_total_ejercicio = 20.0
h_ejercicio = 1.0

# Escriba su solución a partir de aquí.


# 19. Conclusiones

Los métodos numéricos permiten transformar un modelo diferencial en una simulación computacional.

La idea central es:

\[
\boxed{
\text{Modelo}
\rightarrow
\text{Método numérico}
\rightarrow
\text{Simulación}
\rightarrow
\text{Análisis}
}
\]

Euler y RK4 no sustituyen el modelo: **aproximan numéricamente la evolución del modelo que definimos**.

Por ello, una simulación debe analizar tanto:

- la calidad del modelo;
- el método utilizado;
- el tamaño de paso;
- el error;
- y la interpretación de los resultados.
